# Search Algorithms and Case Analysis: Linear vs. Binary Search

In algorithm efficiency, we evaluate performance across **best-case**, **worst-case**, and **average-case** input scenarios.

## Learning Outcomes

- Implement and instrument Linear Search and Binary Search.
- Analyze best, worst, and average-case scenarios for both algorithms.
- Quantify why worst-case complexity is the primary guarantee in software engineering.
- Reproduce the 'Hardware vs. Algorithm' experiment, demonstrating that an $O(\log n)$ algorithm on slow hardware vastly outperforms an $O(n)$ algorithm on a supercomputer.

## 1. Linear Search ($O(n)$)

Linear search scans each element of a sequence sequentially until the target key is found or the list ends.

- **Best Case**: Key is at index 0 $\implies 1$ comparison ($O(1)$).
- **Worst Case**: Key is absent or at index $n - 1$ $\implies n$ comparisons ($O(n)$).
- **Average Case**: Assuming uniform distribution, $\frac{n + 1}{2}$ comparisons ($O(n)$).

In [ ]:
def linear_search_instrumented(arr: list[int], key: int) -> tuple[int, int]:
    """Search for key in arr. Returns (found_index, comparison_count)."""
    comparisons = 0
    for idx, item in enumerate(arr):
        comparisons += 1
        if item == key:
            return idx, comparisons
    return -1, comparisons


sample_arr = [1, 2, 10, 15, 23]
print("Search for 1  (best case)  :", linear_search_instrumented(sample_arr, 1))
print("Search for 10 (average case):", linear_search_instrumented(sample_arr, 10))
print("Search for 23 (worst case)  :", linear_search_instrumented(sample_arr, 23))
print("Search for 34 (not found)   :", linear_search_instrumented(sample_arr, 34))

## 2. Binary Search ($O(\log n)$)

Binary search requires the array to be sorted. It repeatedly bisects the search range in half.

- **Invariant**: If the target key exists in `arr`, it lies within `arr[low..high]`.
- **Best Case**: Key is at the initial midpoint $\implies 1$ comparison ($O(1)$).
- **Worst Case**: Search space halves at each step: $n \to n/2 \to n/4 \to \dots \to 1$.
  Halts in at most $\lfloor \log_2 n \rfloor + 1$ iterations ($O(\log n)$).

In [ ]:
def binary_search_instrumented(arr: list[int], key: int) -> tuple[int, int]:
    """Binary search in sorted array. Returns (found_index, comparison_count)."""
    low = 0
    high = len(arr) - 1
    comparisons = 0

    while low <= high:
        comparisons += 1
        mid = low + (high - low) // 2
        if arr[mid] == key:
            return mid, comparisons
        elif arr[mid] < key:
            low = mid + 1
        else:
            high = mid - 1

    return -1, comparisons


sorted_arr = [1, 2, 10, 15, 23]
print("Binary search for 10 (best midpoint):", binary_search_instrumented(sorted_arr, 10))
print("Binary search for 23 (leaf target)  :", binary_search_instrumented(sorted_arr, 23))
print("Binary search for 34 (absent)       :", binary_search_instrumented(sorted_arr, 34))

## 3. Large-Scale Empirical Comparison

Let us compare linear search and binary search on $n = 1,000,000$ elements.

In [ ]:
import time

large_n = 1_000_000
data = list(range(large_n))  # sorted integers 0 .. 999,999
target = large_n - 1  # worst case: target at the very end

# Measure linear search
t0 = time.perf_counter()
idx_lin, comp_lin = linear_search_instrumented(data, target)
t_lin = time.perf_counter() - t0

# Measure binary search
t0 = time.perf_counter()
idx_bin, comp_bin = binary_search_instrumented(data, target)
t_bin = time.perf_counter() - t0

print(f"Linear search: {comp_lin:,} comparisons | Time: {t_lin * 1000:.4f} ms")
print(f"Binary search: {comp_bin:,} comparisons | Time: {t_bin * 1000:.4f} ms")
print(f"Speedup factor: {comp_lin / comp_bin:,.1f}x fewer comparisons")

## 4. The Algorithm vs. Hardware Simulation

Suppose we search across $n = 10^{10}$ items:
- Linear search: $c_1 n$ operations
- Binary search: $c_2 \log_2 n$ operations (where $\log_2(10^{10}) \approx 33.2$)

Let us compare 4 theoretical cases from the lecture slides:
- **Case 1**: Identical programmer ($c_1=c_2=1$), 1 GHz computer ($10^9$ ops/sec).
  - Linear: $10^{10} / 10^9 = 10$ seconds.
  - Binary: $33.2 / 10^9 \approx 3.32 \times 10^{-8}$ seconds (33 nanoseconds).
- **Case 2**: Best linear programmer ($c_1=1$) vs 'bad' binary programmer ($c_2=50$).
  - Linear: 10 seconds.
  - Binary: $50 \times 33.2 / 10^9 \approx 1.66 \times 10^{-6}$ seconds ($1.66 \mu s$).
- **Case 3**: Fast machine ($10^9$ ops/sec) running linear vs Slow machine ($10^7$ ops/sec) running binary.
  - Linear on fast machine: 10 seconds.
  - Binary on 100x slower machine: $33.2 / 10^7 \approx 3.32 \mu s$.
- **Case 4**: Worst programmer on slow machine ($c_2=50, 10^7$ ops/sec) vs Best linear programmer on fast machine ($c_1=1, 10^9$ ops/sec).
  - Linear on fast machine: 10 seconds.
  - Binary on slow machine: $50 \times 33.2 / 10^7 \approx 1.66 \times 10^{-4}$ seconds (0.166 ms).
  - **Conclusion**: Binary search on the 100x slower computer with 50x sloppier code is still **60,000 times faster** than linear search on the supercomputer!

In [ ]:
import math

n_items = 10**10
log2_n = math.log2(n_items)

# Case 4 calculation
t_linear_fast = (1 * n_items) / 10**9
t_binary_slow = (50 * log2_n) / 10**7

print(f"Linear search (fast machine, best code): {t_linear_fast:.2f} seconds")
print(f"Binary search (slow machine, sloppy code): {t_binary_slow * 1000:.4f} milliseconds")
print(f"Ratio: Binary search is {t_linear_fast / t_binary_slow:,.1f}x faster!")
assert t_linear_fast / t_binary_slow > 50_000